In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

# 🕵️ Your First Anonymization

Detect sensitive entities and replace them with LLM-generated substitutes --
the simplest end-to-end example of Anonymizer.

#### 📚 What you'll learn

- Load a CSV dataset and configure Anonymizer in a few lines
- Preview anonymized results on a small sample before committing to a full run
- Inspect entity detection and replacement with `display_record()`
- Process the full dataset with `run()`

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
- Import the core Anonymizer classes: `Anonymizer`, `AnonymizerConfig`, `AnonymizerInput`, and `Substitute`.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.

In [2]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [3]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, Substitute

In [4]:
anonymizer = Anonymizer()

[18:05:36] [INFO] 🔧 Anonymizer initialized with 3 model configs


[18:05:36] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[18:05:36] [INFO]   |-- ✅ validator: gpt-oss-120b


[18:05:36] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Load data and configure

- `AnonymizerInput` points to your CSV and names the text column. `data_summary`
  gives the LLM context about the kind of text it will process.
- `AnonymizerConfig` with `Substitute()` tells Anonymizer to replace detected
  entities with LLM-generated synthetic values (e.g. fake names, cities, dates).

In [5]:
input_data = AnonymizerInput(
    source="../data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles of individuals",
)

config = AnonymizerConfig(replace=Substitute())

## 👁️ Preview

- `preview()` runs on a small sample so you can iterate quickly.
- Always preview before processing the full dataset -- it's the fastest way
  to catch prompt or config issues early.

In [6]:
preview = anonymizer.preview(config=config, data=input_data, num_records=3)

[18:05:36] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:05:36] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:05:36] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:05:36] [INFO] 🔍 Running entity detection on 3 records


[18:06:33] [INFO]   |-- 📋 Detection complete — 76 entities found across 3 records (0 failed) [56.6s]


[18:06:33] [INFO]   |-- labels: first_name=23, company_name=8, state=6, age=5, occupation=5, city=5, race_ethnicity=4, education_level=4, last_name=3, language=3, political_view=3, religious_belief=2, street_address=2, school_name=1, date_of_birth=1, employment_status=1


[18:06:33] [INFO] 🔄 Running Substitute replacement


[18:06:50] [INFO]   |-- 📋 Replacement complete (0 failed) [17.3s]


[18:06:50] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


## 🔍 Inspect

- `display_record()` shows the original text with highlighted entities,
  the replacement map, and the anonymized output -- all in one view.
- The result dataframe has original and substituted text side-by-side.

In [7]:
preview.display_record(0)

Original,Label,Replacement
40,age,45
Aria,first_name,Lena
Bobby,first_name,Ethan
Christian Democrat,political_view,Libertarian
Colorado,state,Oregon
Colorado Veterinary Clinic,company_name,Oregon Animal Care Center
DVM,education_level,MS
Denver,city,Portland
English,language,Spanish
Jefferson High,school_name,Lincoln High


In [8]:
preview.display_record(1)

Original,Label,Replacement
37,age,41
Bell,last_name,Khan
Edison,city,Boulder
Elena,first_name,Clara
English,language,Spanish
Goddard Space Flight Center,company_name,National Oceanic Research Center
Idilio,first_name,Rafael
Italian,race_ethnicity,Greek
Lina,first_name,Aisha
Marco,first_name,Diego


In [9]:
preview.dataframe

,biography,biography_with_spans,final_entities,biography_replaced
0,"Bobby Watford, a 40‑year‑old Mexican veterinar...",<first_name>Bobby</first_name> <last_name>Watf...,"{'entities': [{'end_position': 5, 'id': 'first...","Ethan Henderson, a 45‑year‑old Filipino animal..."
1,Idilio Bell is a 37‑year‑old astronomer living...,<first_name>Idilio</first_name> <last_name>Bel...,"{'entities': [{'end_position': 6, 'id': 'first...",Rafael Khan is a 41‑year‑old marine biologist ...
2,"Jodi Allison, 36, lives at 204 Bluegrass in Cl...",<first_name>Jodi</first_name> <last_name>Allis...,"{'entities': [{'end_position': 4, 'id': 'first...","Leah Keller, 42, lives at 417 Willowbrook in C..."


## 🚀 Full run

- `run()` processes the entire dataset with the same config you previewed.
- Access the output via `result.dataframe`.

In [10]:
result = anonymizer.run(config=config, data=input_data)
print(result)

[18:06:50] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:06:50] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:06:50] [INFO] 🔍 Running entity detection on 25 records


[18:12:23] [INFO]   |-- 📋 Detection complete — 624 entities found across 25 records (0 failed) [332.5s]


[18:12:23] [INFO]   |-- labels: first_name=154, company_name=68, city=49, occupation=43, education_level=38, state=31, race_ethnicity=30, last_name=26, age=26, religious_belief=26, political_view=25, street_address=23, language=22, county=12, employment_status=10, date_of_birth=9, university_name=6, date=5, institution_name=5, organization_name=5, region=2, school_name=1, college_name=1, university=1, fire_department_name=1, cultural_center_name=1, country=1, gender=1, project_name=1, postcode=1


[18:12:23] [INFO] 🔄 Running Substitute replacement


/Users/lramaswamy/Documents/github/Anonymizer/src/anonymizer/engine/replace/llm_replace_workflow.py:108: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([output_df, passthrough_rows], axis=0)
[18:14:29] [INFO]   |-- 📋 Replacement complete (0 failed) [125.9s]


[18:14:29] [INFO] 🎉 Pipeline complete — 25 records processed, 0 total failures


AnonymizerResult(rows=25, columns=4, trace_columns=21, failed_records=0)


In [11]:
result.dataframe.head()

,biography,biography_with_spans,final_entities,biography_replaced
0,"Bobby Watford, a 40‑year‑old Mexican veterinar...",<first_name>Bobby</first_name> <last_name>Watf...,"{'entities': array([{'end_position': 5, 'id': ...","Ethan Hawthorne, a 45‑year‑old Filipino marine..."
1,Idilio Bell is a 37‑year‑old astronomer living...,<first_name>Idilio</first_name> <last_name>Bel...,"{'entities': array([{'end_position': 6, 'id': ...",Rafael Hawke is a 42‑year‑old geophysicist liv...
2,"Jodi Allison, 36, lives at 204 Bluegrass in Cl...",<first_name>Jodi</first_name> <last_name>Allis...,"{'entities': array([{'end_position': 4, 'id': ...","Sofia Keller, 42, lives at 317 Maplewood in Fa..."
3,James Mills is a 69‑year‑old paramedic who liv...,<first_name>James</first_name> <last_name>Mill...,"{'entities': array([{'end_position': 5, 'id': ...",Robert Hawkins is a 57‑year‑old firefighter wh...
4,Nancy Burton is a 21‑year‑old cashier who live...,<first_name>Nancy</first_name> <last_name>Burt...,"{'entities': array([{'end_position': 5, 'id': ...",Leah Kline is a 22‑year‑old warehouse associat...


## ⏭️ Next steps

- **[🔍 Inspecting Detected Entities](02_inspecting_detected_entities.ipynb)** --
  dig into what the detection pipeline found and debug quality.
- **[🎯 Choosing a Replacement Strategy](03_choosing_a_replacement_strategy.ipynb)** --
  compare Redact, Annotate, Hash, and Substitute side-by-side.
- **[✏️ Rewriting Biographies](04_rewriting_biographies.ipynb)** --
  generate privacy-safe paraphrases instead of token-level replacements.